## Semantic Movie Recommendation using Sentence-BERT (SBERT)

This notebook implements a semantic content-based movie recommendation system using a pre-trained Sentence-BERT (SBERT) model. Unlike the Bag of Words approach, SBERT generates dense contextual embeddings that capture the semantic meaning of movie descriptions. Cosine similarity is then used to identify movies with similar content.

In [1]:
## Import Required Libraries
import pandas as pd
import numpy as np 
import pickle

In [2]:
# Load the preprocessed movie dataset.
new_df = pickle.load(open("new_df.pkl", "rb")) 

In [3]:
# Preview the processed dataset.
new_df.head() 

,title,movie_id,tags
0,Avatar,19995,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,285,"captain barbossa, long believed to be dead, ha..."
2,Spectre,206647,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,49026,following the death of district attorney harve...
4,John Carter,49529,"john carter is a war-weary, former military ca..."


In [4]:
new_df.iloc[0]["tags"] 

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

### Load the Pre-trained SBERT Model

Sentence-BERT (SBERT) is a pre-trained transformer model designed to generate dense sentence embeddings. Unlike Bag of Words, SBERT understands the contextual meaning of text, enabling more meaningful similarity comparisons between movies.

In [5]:
# Import the SentenceTransformer class.
from sentence_transformers import SentenceTransformer

In [6]:
# Load the pre-trained all-MiniLM-L6-v2 model.
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Generate Sentence Embeddings

Each movie is converted into a 384-dimensional semantic embedding using the pre-trained SBERT model. These embeddings capture the contextual meaning of the movie tags rather than simply counting word occurrences.

In [7]:
# Generate semantic embeddings for all movies.
embeddings = model.encode(new_df["tags"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

In [8]:
# Display the shape of the generated semantic embeddings.
embeddings.shape

(4806, 384)

The embedding matrix has a shape of **(4806, 384)**. Each of the **4806 rows** represents a movie, while the **384 columns** represent the semantic embedding generated by the pre-trained SBERT model. Unlike Bag of Words, where each feature corresponds to a specific word, these 384 dimensions do not have explicit meanings. Instead, they collectively encode the contextual and semantic information of the movie tags, allowing movies with similar meanings to have similar vector representations.

### Compute Cosine Similarity

Cosine similarity measures the similarity between the semantic embeddings of movies. Movies with higher cosine similarity scores are considered more semantically related and are recommended together.

In [9]:
# Import the cosine similarity function.
from sklearn.metrics.pairwise import cosine_similarity

# Compute the cosine similarity matrix.
similarity = cosine_similarity(embeddings)

### Recommendation Function

The recommendation function retrieves the five most semantically similar movies for a given movie title using the computed cosine similarity scores.

In [10]:
# Generate the top five movie recommendations.
def recommend(movie): 
    movie_index = new_df[new_df["title"] == movie].index[0] 

    similar_items = sorted(list(enumerate(similarity[movie_index])), key = lambda x: x[1], reverse=True)[1:6] 

    recommendations = [] 
    
    for el in similar_items: 
        recommendations.append((new_df.iloc[el[0]].title, new_df.iloc[el[0]].movie_id)) 
    return recommendations 

In [11]:
# Generate recommendations for a sample movie.
recommend("Avatar")

[('Serenity', np.int64(16320)),
 ('Alien: Resurrection', np.int64(8078)),
 ('Aliens', np.int64(679)),
 ('The Inhabited Island', np.int64(16911)),
 ('Invaders from Mars', np.int64(31909))]

In [12]:
# Save the similarity matrix.
pickle.dump(similarity, open("similarity_SBERT.pkl", "wb")) 